# Bulk RNA-seq: supplementary analyses

Figures S2, S3, S4, S14; Tables S2, S4 (gene lists for STRING-DB)

Environment: `python_env` (`envs/python_env.yml`)

Inputs:
- `out_dir/bulk_deseq2/*.csv` (from `bulk_RNAseq_1_DESeq2.ipynb`)
- `bulk_RNAseq_htseq_raw_counts.csv` (GSE348942), in `data_dir/GSE348942/`
- `data/bulk_RNAseq_sample_metadata.csv`
- `data/bulk_MS2_strand_depth.csv.gz`, `data/bulk_mapped_reads.csv` (from `preprocessing/bulk_MS2_coverage.py`)

In [ ]:
from pathlib import Path
import yaml
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

plt.rcParams["pdf.fonttype"] = 42

ROOT = Path("..")
config = yaml.safe_load(open(ROOT / "config.yaml"))
DATA_DIR = Path(config["data_dir"]) / "GSE348942"
OUT_DIR = Path(config["out_dir"])
DE_DIR = OUT_DIR / "bulk_deseq2"
FIG_DIR = OUT_DIR / "figures"
TAB_DIR = OUT_DIR / "tables"
SRC_DIR = OUT_DIR / "source_data"
for d in (FIG_DIR, TAB_DIR, SRC_DIR):
    d.mkdir(parents=True, exist_ok=True)

palette = ["#1F77B4", "#FF7F0E"]
TIMES = [0, 5, 10, 15, 45, 180]
MS2 = ["MS2", "MS2_negative"]

meta = pd.read_csv(ROOT / "data" / "bulk_RNAseq_sample_metadata.csv", index_col="sample")


def save_fig(fig, name, dpi):
    fig.savefig(FIG_DIR / f"{name}.png", dpi=dpi, bbox_inches="tight")
    fig.savefig(FIG_DIR / f"{name}.pdf", bbox_inches="tight")


def load_de(name):
    return pd.read_csv(DE_DIR / f"{name}.csv", index_col=0)


def significant(de, lfc, padj=0.05):
    de = de[~de.index.isin(MS2)]
    return de[(de.padj < padj) & (de.log2FoldChange.abs() > lfc)]

## Figure S2

In [ ]:
depth = pd.read_csv(ROOT / "data" / "bulk_MS2_strand_depth.csv.gz")
mapped = pd.read_csv(ROOT / "data" / "bulk_mapped_reads.csv")
BIN_SIZE = 300

depth = depth.join(meta[["time", "infected"]], on="sample")
mapped = mapped.join(meta[["time", "infected"]], on="sample")
depth = depth[depth.infected == "yes"]
mapped = mapped[mapped.infected == "yes"]

mean_depth = depth.groupby(["time", "infected", "strand", "position"], as_index=False)["coverage"].mean()
mean_depth["bin"] = ((mean_depth.position / BIN_SIZE).round() * BIN_SIZE).astype(int)
binned = mean_depth.groupby(["time", "infected", "strand", "bin"], as_index=False)["coverage"].sum()

lib = mapped.groupby(["time", "infected"], as_index=False)["mapped_reads"].mean()
binned = binned.merge(lib, on=["time", "infected"], how="left", validate="many_to_one")
assert binned.mapped_reads.notna().all()
binned["CPM"] = binned.coverage / binned.mapped_reads * 1e6

plot_data = binned.copy()
plot_data["strand"] = plot_data.strand.str.capitalize()
plot_data["time"] = pd.Categorical(plot_data.time, categories=TIMES, ordered=True)

g = sns.FacetGrid(plot_data, row="strand", col="time", col_order=TIMES, row_order=["Forward", "Reverse"],
                  height=3, sharex=True, sharey=False, margin_titles=True, despine=False)
g.map_dataframe(sns.lineplot, x="bin", y="CPM", style="strand",
                dashes={"Forward": "", "Reverse": (2, 2)}, color="tab:orange", legend=False)
g.set_titles(row_template="{row_name}", col_template="{col_name}")

for ax in g.axes.flat:
    ax.grid(alpha=0.2)
    ax.tick_params(axis="y", labelsize=12)
    ax.yaxis.get_offset_text().set_fontsize(12)
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(1)
for ax, t in zip(g.axes[0], TIMES):
    ax.set_title(f"{t} minutes", fontsize=20, y=1.1)
for ax in g.axes[:, -1]:
    for label in ax.texts:
        label.set_fontsize(18)

g.set_axis_labels("", "")
g.figure.supxlabel("MS2 position (bp)", fontsize=18, y=0.0)
g.figure.supylabel("Read depth per million", fontsize=18, x=0.01)
plt.subplots_adjust(hspace=0.3, wspace=0.4, bottom=0.14, left=0.07)

for i, row in enumerate(g.axes):
    for ax in row:
        ax.set_xticks([0, 1500, 3000])
        ax.tick_params(axis="x", which="both", bottom=True, labelbottom=(i == len(g.axes) - 1), labelsize=15)

save_fig(g.figure, "FigS2", 300)
plt.show()
plot_data.to_csv(SRC_DIR / "FigS2_data.csv", index=False)

## Figure S3

In [ ]:
LFC = 1.5
LABEL_45_INF = "45 vs. 15\ninfected"
LABEL_45_CTRL = "45 vs. 15\ncontrol"
LABEL_180 = "180 vs. 15"
COMPARISONS = [LABEL_45_INF, LABEL_45_CTRL, LABEL_180]


def up_down(de):
    return de.index[de.log2FoldChange > 0].tolist(), de.index[de.log2FoldChange < 0].tolist()


groups = {}
for t in TIMES:
    groups[t] = up_down(significant(load_de(f"infected_vs_control_{t}min"), LFC))
groups[LABEL_45_INF] = up_down(significant(load_de("infected_45min_vs_15min"), LFC))
groups[LABEL_45_CTRL] = up_down(significant(load_de("control_45min_vs_15min"), LFC))

inf_up, inf_down = up_down(significant(load_de("infected_180min_vs_15min"), LFC))
ctrl_up, ctrl_down = up_down(significant(load_de("control_180min_vs_15min"), LFC))
groups[LABEL_180] = ([g for g in inf_up if g not in set(ctrl_up)],
                     [g for g in inf_down if g not in set(ctrl_down)])

counts = pd.DataFrame({"group": [str(k) for k in groups],
                       "up": [len(v[0]) for v in groups.values()],
                       "down": [len(v[1]) for v in groups.values()]})

spacing = 1.8
width = 0.4
colors = {"down": "#D6D6D6", "up": "#959596"}
all_groups = TIMES + COMPARISONS

x_pos = {t: i for i, t in enumerate(TIMES)}
first_x = len(TIMES) - 1 + spacing * 1.25
for j, c in enumerate(COMPARISONS):
    x_pos[c] = first_x + j * spacing

fig, ax = plt.subplots(figsize=(6.6, 2.8))
for i, (d, offset) in enumerate([("down", -width / 2), ("up", width / 2)]):
    ax.bar([x_pos[g] + offset for g in all_groups], [len(groups[g][1 - i]) for g in all_groups],
           width=width, color=colors[d], edgecolor="white", label=d)

ax.set_xticks([x_pos[g] for g in all_groups])
ax.set_xticklabels([str(g) for g in all_groups], fontsize=10)
ax.tick_params(axis="y", labelsize=10)
ax.set_xlim(-0.6, x_pos[LABEL_180] + 0.6)
ax.axvline(x=(len(TIMES) - 1 + first_x) / 2, color="black", linestyle="--", linewidth=1)
ax.axvline(x=(x_pos[LABEL_45_CTRL] + x_pos[LABEL_180]) / 2, color="black", linestyle="--", linewidth=1)
ax.legend(title="Differential Expression", loc="upper center", bbox_to_anchor=(0.5, -0.36),
          ncol=2, frameon=False, fontsize=10, title_fontsize=10)
ax.set_ylabel("Number of differentially\nexpressed genes", fontsize=12)
ax.set_xlabel("Time (minutes post infection)", fontsize=12)

save_fig(fig, "FigS3", 800)
plt.show()
counts.to_csv(SRC_DIR / "FigS3_data.csv", index=False)
counts

## Figure S4

In [ ]:
LFC_EARLY = np.log2(1.5)

early = pd.DataFrame({t: significant(load_de(f"infected_vs_control_{t}min"), LFC_EARLY).log2FoldChange
                      for t in [0, 5, 10, 15]})
early = early[(early > 0).sum(axis=1) >= 2]

cmap = sns.diverging_palette(260, 15, sep=10, as_cmap=True)
cg = sns.clustermap(early.fillna(0), center=0, cmap=cmap, vmin=-2, vmax=2, col_cluster=False,
                    cbar_pos=(0.05, 0.35, 0.05, 0.18), figsize=(6, 5), cbar_kws={"label": "Log2 fold change"})
cg.ax_row_dendrogram.set_visible(False)
cg.ax_heatmap.set_yticklabels(cg.ax_heatmap.get_yticklabels(), rotation=0)
cg.ax_heatmap.set_xlabel("Time (minutes post infection)", labelpad=12)
cg.ax_heatmap.set_ylabel("\ngene")

save_fig(cg.figure, "FigS4", 800)
plt.show()
early.rename_axis("gene").to_csv(SRC_DIR / "FigS4_data.csv")
len(early)

## Figure S14

In [ ]:
raw = pd.read_csv(DATA_DIR / "bulk_RNAseq_htseq_raw_counts.csv", index_col=0)
print("htseq summary rows:", [g for g in raw.index if g.startswith("__")])

pct = (raw.loc[MS2] / raw.sum(axis=0) * 100).T
pct = pct.join(meta[["time", "infected"]]).reset_index(names="sample")
pct = pct.melt(id_vars=["sample", "time", "infected"], var_name="gene", value_name="percent")
pct = pct[pct.infected == "no"].sort_values("time")
pct["gene"] = np.where(pct.gene == "MS2", "MS2 ($+$)", "MS2 ($-$)")

spacing = {0: 0.0, 5: 1.1, 10: 1.9, 15: 2.6, 45: 4.8, 180: 7.6}
pct["x"] = pct.time.map(spacing)

style = {"font.size": 12, "axes.labelsize": 13, "axes.titlesize": 14,
         "xtick.labelsize": 11, "ytick.labelsize": 11, "legend.fontsize": 11}
with plt.rc_context(style):
    fig, ax = plt.subplots(figsize=(6.0, 2.8))
    sns.lineplot(data=pct, x="x", y="percent", style="gene", style_order=["MS2 ($+$)", "MS2 ($-$)"],
                 markers=True, ax=ax, color=palette[0], dashes=["", (2, 2)], errorbar="se")
    ax.grid(alpha=0.2)
    ax.legend(loc="upper left", bbox_to_anchor=(1, 0.8), ncol=1, frameon=False, title="")
    ax.set_ylabel("% of reads")
    ax.set_xlabel("Time (minutes post infection)")
    ax.set_xticks(list(spacing.values()))
    ax.set_xticklabels([str(t) for t in spacing])
    ax.set_xlim(-0.4, 8.0)
    save_fig(fig, "FigS14", 800)
    plt.show()

pct.drop(columns="x").to_csv(SRC_DIR / "FigS14_data.csv", index=False)

## Tables S2 and S4

Gene lists used as input for STRING-DB (GO biological process). Paralog suffixes (e.g. `_1`) are removed for STRING.

In [ ]:
LFC_STRING = np.log2(1.5)


def string_names(genes):
    return [g.split("_")[0] if "_" in g and not g.startswith("LELOBEKK") else g for g in genes]


s2 = significant(load_de("infected_vs_control_45min"), LFC_STRING)
s2_genes = s2.index[s2.log2FoldChange > 0].tolist()

inf = significant(load_de("infected_180min_vs_15min"), LFC_STRING)
ctrl = significant(load_de("control_180min_vs_15min"), LFC_STRING)
ctrl_up = set(ctrl.index[ctrl.log2FoldChange > 0])
s4_genes = [g for g in inf.index[inf.log2FoldChange > 0] if g not in ctrl_up]

pd.Series(string_names(s2_genes)).to_csv(TAB_DIR / "TableS2_input_genes.txt", index=False, header=False)
pd.Series(string_names(s4_genes)).to_csv(TAB_DIR / "TableS4_input_genes.txt", index=False, header=False)
{"TableS2": len(s2_genes), "TableS4": len(s4_genes)}